# Phase 2: Data Preprocessing & Reconciliation Check
## Project: Workforce Attrition Patterns and Risk Hotspot Analysis
**Scope:** Phase 2 – Verification of Analysis-Ready Dataset & Feature Derivations

> **DISCLAIMER:** The dataset provenance is UNVERIFIED. This data is analyzed as a public/synthetic HR benchmark dataset. Findings must not be described as real Palo Alto Networks employment records or proprietary corporate data. No modeling or EDA conclusions are drawn in this verification notebook.

In [1]:
import sys
from pathlib import Path

# Ensure project src is in python path
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
from IPython.display import display

from workforce_intel.config import (
    PROCESSED_DATA_FILE, PROCESSED_SCHEMA_FILE, RAW_DATA_FILE
)
from workforce_intel.io import load_processed, load_raw
from workforce_intel.validation import reconciliation_audit

print('Libraries and verification modules imported successfully.')

Libraries and verification modules imported successfully.


## 1. Load Datasets & Execute Reconciliation
Verify row-level and cell-for-cell identity between raw and processed datasets.

In [2]:
raw_df, delimiter = load_raw(RAW_DATA_FILE)
processed_df = load_processed(PROCESSED_DATA_FILE, PROCESSED_SCHEMA_FILE)

print(f'Raw Shape: {raw_df.shape}')
print(f'Processed Shape: {processed_df.shape}')

recon = reconciliation_audit(raw_df, processed_df)
display(recon['reconciliation_table'])

Raw Shape: (1470, 31)
Processed Shape: (1470, 47)


,Metric,Value,Expected,Status
0,Raw Dataset Rows,1470,1470,PASS
1,Processed Dataset Rows,1470,1470,PASS
2,Rows Dropped,0,0,PASS
3,Original 31 Columns Exact Match,100% Identical,100% Identical,PASS
4,RowID Sequence [0 .. n-1],Valid & Continuous,Valid & Continuous,PASS
5,Derived Columns Count,15,15,PASS
6,Total Processed Columns,47,47,PASS


## 2. Derived Feature Specifications
Audit the 15 derived features, non-null counts, and value domains.

In [3]:
display(recon['derived_feature_audit'])

,derived_column,dtype,non_null_count,null_count,domain_summary
0,AgeGroup,category,1470,0,"4 categories: ['18-25', '26-35', '36-45', '46-..."
1,TenureBand,category,1470,0,"5 categories: ['0-1', '2-3', '4-5', '6-10', '1..."
2,EarlyTenure,int64,1470,0,Range [0 .. 1]
3,CareerStage,category,1470,0,"4 categories: ['0-5', '6-10', '11-20', '21+']"
4,RoleTenureBand,category,1470,0,"4 categories: ['0-1', '2-3', '4-7', '8+']"
5,ManagerTenureBand,category,1470,0,"4 categories: ['0-1', '2-3', '4-7', '8+']"
6,PromotionGapBand,category,1470,0,"4 categories: ['0', '1-2', '3-5', '6+']"
7,DistanceBand,category,1470,0,"4 categories: ['1-5', '6-10', '11-20', '21-29']"
8,PriorExperienceYears,int64,1470,0,Range [0 .. 33]
9,HasPriorExperience,int64,1470,0,Range [0 .. 1]


## 3. Subgroup & Bucket Sizes Audit (n < 30 Rule)
Inspect distribution counts and flags for all derived categorical bands and indicators.

In [4]:
bucket_df = recon['bucket_sizes_table']
display(bucket_df)

small_groups = bucket_df[bucket_df['is_small_group']]
print(f'Small subgroups (n < 30) detected: {len(small_groups)}')

,column,level,count,percentage,is_small_group
0,AgeGroup,26-35,606,41.22,False
1,AgeGroup,36-45,468,31.84,False
2,AgeGroup,46-60,273,18.57,False
3,AgeGroup,18-25,123,8.37,False
4,TenureBand,6-10,448,30.48,False
5,TenureBand,4-5,306,20.82,False
6,TenureBand,2-3,255,17.35,False
7,TenureBand,11+,246,16.73,False
8,TenureBand,0-1,215,14.63,False
9,EarlyTenure,0,1255,85.37,False


Small subgroups (n < 30) detected: 0


## 4. Zero-Denominator Rule & Missingness Audit
Verify that only RoleTenureShare and PromotionGapShare contain NaNs (exactly 44 rows where YearsAtCompany == 0).

In [5]:
nan_df = recon['nan_audit_table']
display(nan_df[nan_df['nan_count'] > 0])

assert (nan_df.loc[~nan_df['column'].isin(['RoleTenureShare', 'PromotionGapShare']), 'nan_count'] == 0).all()
print('Zero unexpected NaNs across all other 45 columns verified.')

,column,nan_count,nan_pct,expected_nan_count,status
42,RoleTenureShare,44,2.99,44,PASS
43,PromotionGapShare,44,2.99,44,PASS


Zero unexpected NaNs across all other 45 columns verified.
